## 1. Initialize Project Environment
Import libraries for data manipulation, normalization, and filtering.

In [10]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)

pandas 2.2.3
numpy 2.1.3


## 2. Define Configuration Parameters
Centralize paths, filtering thresholds, and normalization options for reproducibility.

In [11]:
@dataclass
class PrepConfig:
    handle: str = "AndreiCod"
    n_genes: int = 200
    n_samples: int = 30
    variance_threshold: float = 0.1  # Lower threshold to retain more genes
    export_dir: Path = Path("artifacts")
    seed: int = 42

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = PrepConfig()
CONFIG.describe()

{'handle': 'AndreiCod',
 'n_genes': 200,
 'n_samples': 30,
 'variance_threshold': 0.1,
 'export_dir': 'artifacts',
 'seed': 42}

## 3. Generate Synthetic TP53-Associated Expression Data
Create a realistic RNA-Seq dataset simulating TP53-pathway genes with tumor/normal samples and biological modules.

In [12]:
def create_tp53_expression_data(cfg: PrepConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Create synthetic expression matrix simulating TP53-associated patterns.
    Creates 5 biologically-inspired modules:
    - Module 1: TP53 core pathway (apoptosis)
    - Module 2: Cell cycle regulators
    - Module 3: DNA damage response
    - Module 4: Metabolic genes
    - Module 5: Housekeeping/background genes
    """
    np.random.seed(cfg.seed)

    n_genes = cfg.n_genes
    n_samples = cfg.n_samples

    # Sample metadata: half tumor, half normal
    n_tumor = n_samples // 2
    n_normal = n_samples - n_tumor

    sample_ids = [f"Sample_{i + 1}" for i in range(n_samples)]
    conditions = ["Tumor"] * n_tumor + ["Normal"] * n_normal
    metadata = pd.DataFrame({"SampleID": sample_ids, "Condition": conditions})

    # Gene names with biological relevance
    tp53_pathway = [
        "TP53",
        "MDM2",
        "BAX",
        "PUMA",
        "NOXA",
        "BID",
        "APAF1",
        "CASP9",
        "CASP3",
        "BCL2",
        "BCL2L1",
        "MCL1",
        "BIRC5",
        "XIAP",
        "CYCS",
        "DIABLO",
        "ENDOG",
        "AIF",
        "HTRA2",
        "PARP1",
    ]
    cell_cycle = [
        "CDKN1A",
        "CDKN2A",
        "RB1",
        "E2F1",
        "E2F2",
        "CCND1",
        "CCNE1",
        "CDK2",
        "CDK4",
        "CDK6",
        "CCNB1",
        "CDC25A",
        "CDC25C",
        "PLK1",
        "AURKA",
        "AURKB",
        "BUB1",
        "MAD2L1",
        "CHEK1",
        "CHEK2",
    ]
    dna_damage = [
        "ATM",
        "ATR",
        "BRCA1",
        "BRCA2",
        "RAD51",
        "XRCC1",
        "PARP2",
        "MLH1",
        "MSH2",
        "MSH6",
        "OGG1",
        "XPA",
        "XPC",
        "ERCC1",
        "ERCC2",
        "POLE",
        "POLD1",
        "FEN1",
        "LIG1",
        "LIG3",
    ]
    metabolic = [
        "HIF1A",
        "LDHA",
        "PKM",
        "GLUT1",
        "HK2",
        "PFKFB3",
        "SCO2",
        "TIGAR",
        "GLS2",
        "PTEN",
        "AKT1",
        "MTOR",
        "AMPK",
        "PGC1A",
        "SIRT1",
        "SIRT3",
        "UCP2",
        "CPT1A",
        "ACACA",
        "FASN",
    ]

    # Fill remaining with generic genes
    n_known = len(tp53_pathway) + len(cell_cycle) + len(dna_damage) + len(metabolic)
    n_housekeeping = n_genes - n_known
    housekeeping = [f"GENE_{i + 1}" for i in range(n_housekeeping)]

    all_genes = tp53_pathway + cell_cycle + dna_damage + metabolic + housekeeping
    all_genes = all_genes[:n_genes]

    # Create expression patterns
    t = np.linspace(0, 2 * np.pi, n_samples)

    # Module-specific base patterns
    # Module 1 (TP53 pathway): upregulated in tumors
    pattern1_tumor = np.concatenate(
        [np.random.normal(150, 20, n_tumor), np.random.normal(50, 15, n_normal)]
    )
    # Module 2 (Cell cycle): highly variable in tumors
    pattern2_tumor = np.concatenate(
        [np.random.normal(200, 40, n_tumor), np.random.normal(80, 10, n_normal)]
    )
    # Module 3 (DNA damage): moderately elevated in tumors
    pattern3_tumor = np.concatenate(
        [np.random.normal(100, 25, n_tumor), np.random.normal(60, 10, n_normal)]
    )
    # Module 4 (Metabolic): shifted metabolic profile
    pattern4_tumor = np.concatenate(
        [np.random.normal(120, 30, n_tumor), np.random.normal(90, 15, n_normal)]
    )
    # Module 5 (Housekeeping): stable across conditions
    pattern5_stable = np.random.normal(100, 10, n_samples)

    expression_data = []
    gene_modules = []

    for i, gene in enumerate(all_genes):
        if gene in tp53_pathway:
            base_pattern = pattern1_tumor.copy()
            module = "TP53_pathway"
        elif gene in cell_cycle:
            base_pattern = pattern2_tumor.copy()
            module = "Cell_cycle"
        elif gene in dna_damage:
            base_pattern = pattern3_tumor.copy()
            module = "DNA_damage"
        elif gene in metabolic:
            base_pattern = pattern4_tumor.copy()
            module = "Metabolic"
        else:
            base_pattern = pattern5_stable.copy()
            module = "Housekeeping"

        # Add gene-specific noise
        noise = np.random.randn(n_samples) * 10
        gene_expr = base_pattern + noise + np.random.rand() * 20
        gene_expr = np.maximum(gene_expr, 1)  # Ensure positive values
        expression_data.append(gene_expr)
        gene_modules.append({"Gene": gene, "TrueModule": module})

    expr_df = pd.DataFrame(expression_data, index=all_genes, columns=sample_ids)
    module_df = pd.DataFrame(gene_modules)

    logging.info(
        f"Created expression matrix: {expr_df.shape[0]} genes x {expr_df.shape[1]} samples"
    )
    logging.info(f"Conditions: {n_tumor} Tumor, {n_normal} Normal")

    return expr_df, metadata, module_df


expr_raw, metadata, true_modules = create_tp53_expression_data(CONFIG)
print(f"\nExpression matrix shape: {expr_raw.shape}")
print(f"\nSample metadata:")
metadata.head(10)

[INFO] Created expression matrix: 200 genes x 30 samples
[INFO] Conditions: 15 Tumor, 15 Normal



Expression matrix shape: (200, 30)

Sample metadata:


,SampleID,Condition
0,Sample_1,Tumor
1,Sample_2,Tumor
2,Sample_3,Tumor
3,Sample_4,Tumor
4,Sample_5,Tumor
5,Sample_6,Tumor
6,Sample_7,Tumor
7,Sample_8,Tumor
8,Sample_9,Tumor
9,Sample_10,Tumor


In [13]:
# View true module assignments
print("True module distribution:")
true_modules["TrueModule"].value_counts()

True module distribution:


TrueModule
Housekeeping    120
TP53_pathway     20
Cell_cycle       20
DNA_damage       20
Metabolic        20
Name: count, dtype: int64

## 4. Preprocessing: Log Transformation and Variance Filtering
Apply log2(x+1) transformation and filter low-variance genes to focus on informative features.

In [14]:
def log_transform(df: pd.DataFrame) -> pd.DataFrame:
    """Apply log2(x+1) transformation to expression data."""
    df_log = np.log2(df + 1)
    logging.info(f"Applied log2(x+1) transformation")
    logging.info(
        f"  Value range: {df_log.values.min():.2f} - {df_log.values.max():.2f}"
    )
    return df_log


def filter_low_variance(
    df: pd.DataFrame, threshold: float
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Filter genes with variance below threshold."""
    variances = df.var(axis=1)
    variance_df = pd.DataFrame({"Gene": df.index, "Variance": variances}).sort_values(
        "Variance", ascending=False
    )

    mask = variances >= threshold
    df_filtered = df[mask]

    logging.info(f"Variance filtering (threshold={threshold}):")
    logging.info(f"  Genes before: {df.shape[0]}")
    logging.info(f"  Genes after: {df_filtered.shape[0]}")
    logging.info(f"  Removed: {df.shape[0] - df_filtered.shape[0]}")

    return df_filtered, variance_df


# Apply preprocessing
expr_log = log_transform(expr_raw)
expr_filtered, variance_stats = filter_low_variance(expr_log, CONFIG.variance_threshold)

print(f"\nFiltered expression matrix shape: {expr_filtered.shape}")

[INFO] Applied log2(x+1) transformation
[INFO]   Value range: 3.95 - 8.28
[INFO] Variance filtering (threshold=0.1):
[INFO]   Genes before: 200
[INFO]   Genes after: 67
[INFO]   Removed: 133



Filtered expression matrix shape: (67, 30)


In [15]:
# Variance statistics
print("Top 10 highest variance genes:")
variance_stats.head(10)

Top 10 highest variance genes:


,Gene,Variance
BID,BID,1.076962
BCL2L1,BCL2L1,0.956269
AIF,AIF,0.953538
DIABLO,DIABLO,0.910451
CASP9,CASP9,0.901406
PUMA,PUMA,0.849103
BAX,BAX,0.827353
XIAP,XIAP,0.814570
TP53,TP53,0.805506
BCL2,BCL2,0.790895


In [16]:
# Summary statistics of preprocessed data
summary_stats = {
    "raw_genes": expr_raw.shape[0],
    "raw_samples": expr_raw.shape[1],
    "filtered_genes": expr_filtered.shape[0],
    "variance_threshold": CONFIG.variance_threshold,
    "genes_removed": expr_raw.shape[0] - expr_filtered.shape[0],
    "mean_expression": expr_filtered.values.mean(),
    "std_expression": expr_filtered.values.std(),
}
pd.DataFrame([summary_stats])

,raw_genes,raw_samples,filtered_genes,variance_threshold,genes_removed,mean_expression,std_expression
0,200,30,67,0.1,133,6.688877,0.70348


## 5. Validate with Unit Tests
Sanity checks for the preprocessing pipeline.

In [17]:
def test_log_transform():
    test_df = pd.DataFrame(
        {"A": [0, 1, 3, 7, 15]}, index=["g1", "g2", "g3", "g4", "g5"]
    )
    result = log_transform(test_df)
    assert result["A"].iloc[0] == 0  # log2(0+1) = 0
    assert result["A"].iloc[1] == 1  # log2(1+1) = 1
    assert np.isclose(result["A"].iloc[2], 2)  # log2(3+1) = 2


def test_variance_filter():
    test_df = pd.DataFrame(
        {"S1": [1, 10, 100], "S2": [1, 20, 200], "S3": [1, 30, 300]},
        index=["low_var", "med_var", "high_var"],
    )
    filtered, _ = filter_low_variance(test_df, threshold=50)
    assert "high_var" in filtered.index


test_log_transform()
test_variance_filter()
print("All preprocessing tests passed.")

[INFO] Applied log2(x+1) transformation
[INFO]   Value range: 0.00 - 4.00
[INFO] Variance filtering (threshold=50):
[INFO]   Genes before: 3
[INFO]   Genes after: 2
[INFO]   Removed: 1


All preprocessing tests passed.


## 6. Export Results
Save preprocessed expression matrix and metadata for downstream tasks.

In [18]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save preprocessed expression matrix
expr_filtered.to_csv(EXPORT_DIR / "task1_expression_preprocessed.csv")
print(
    f"[OK] Preprocessed expression saved to: {EXPORT_DIR / 'task1_expression_preprocessed.csv'}"
)

# Save raw expression for reference
expr_raw.to_csv(EXPORT_DIR / "task1_expression_raw.csv")
print(f"[OK] Raw expression saved to: {EXPORT_DIR / 'task1_expression_raw.csv'}")

# Save metadata
metadata.to_csv(EXPORT_DIR / "task1_sample_metadata.csv", index=False)
print(f"[OK] Sample metadata saved to: {EXPORT_DIR / 'task1_sample_metadata.csv'}")

# Save true module labels for validation
true_modules.to_csv(EXPORT_DIR / "task1_true_modules.csv", index=False)
print(f"[OK] True module labels saved to: {EXPORT_DIR / 'task1_true_modules.csv'}")

# Save variance statistics
variance_stats.to_csv(EXPORT_DIR / "task1_variance_stats.csv", index=False)
print(f"[OK] Variance statistics saved to: {EXPORT_DIR / 'task1_variance_stats.csv'}")

[OK] Preprocessed expression saved to: artifacts/task1_expression_preprocessed.csv
[OK] Raw expression saved to: artifacts/task1_expression_raw.csv
[OK] Sample metadata saved to: artifacts/task1_sample_metadata.csv
[OK] True module labels saved to: artifacts/task1_true_modules.csv
[OK] Variance statistics saved to: artifacts/task1_variance_stats.csv
